In [ ]:
#imports required
import json
from graphviz import Digraph
from IPython.display import Image

In [ ]:
#helper functions to conver the nfa to dfa and then minimize the dfa
def load_nfa(filename):
    with open(filename, 'r') as f:
        return json.load(f)
    #split the input and puts it into a list
def get_targets(transition_value):
    #standardize how we read transistions
    if isinstance(transition_value,str):
        return [s.strip() for s in transition_value.split(',') if s.strip()]
    elif isinstance(transition_value,list):
        return transition_value
    return []
#does dfs to see all possible transitions with epsilon
def epsilon_closure(states, nfa):
    #compute the epsilon closure of a set of states
    closure = set(states)
    stack = list(states)

    #list of all eps closure due to json errors
    eps_keys = ['ε', 'epsilon', '', 'Îµ', '\u00ce\u00b5']
    while stack:
        curr = stack.pop()
        if curr in nfa:
            eps_transitions = []
            for key in eps_keys:
                if key in nfa[curr]:
                    eps_transitions.extend(get_targets(nfa[curr][key]))
            for target in eps_transitions:
                if target not in closure:
                    closure.add(target)
                    stack.append(target)
    return frozenset(closure)


In [ ]:
#convert the nfa to dfa using subset construction
def nfa_to_dfa(nfa):

    #extract the alphabet from the nfa knows all possible transitions
    alphabet = set()
    for state,data in nfa.items():
        if state == 'startingState':
            continue
        for key in data:
            #epsilon isnt part of the alphabet so we ignore it
            if key not in ('isTerminatingState', 'epsilon', '', 'ε', 'Îµ', '\u00ce\u00b5'):
                alphabet.add(key)
    #Setup the start state
    start_state = nfa['startingState']
    start_closure = epsilon_closure({start_state}, nfa)

    dfa_state = {start_closure: "D0"}
    transitions={}
    accepting = set()
    #we use a bfs to process new dfa states
    unprocessed = [start_closure]
    state_count = 1
    # Process each DFA state
    while unprocessed:
        curr_set= unprocessed.pop(0)
        curr_name = dfa_state[curr_set]
        transitions[curr_name] = {}
        #determin if it is a terminating state or not
        if any(nfa.get(s,{}).get('isTerminatingState', False) for s in curr_set):
            accepting.add(curr_name)
        #test all letter in the alphabet
        for symbol in alphabet:
            next_set= set()
            #from current state where does this symbol take us
            for s in curr_set:
                trans = nfa.get(s, {}).get(symbol)
                if trans:
                    next_set.update(get_targets(trans))
            #compute the epsilon closure  of those states
            next_closure = frozenset(epsilon_closure(next_set, nfa))
            #checks if we have this state before
            if next_closure not in dfa_state:
                #new give new name
                dfa_state[next_closure] = f"D{state_count}"
                state_count += 1
                unprocessed.append(next_closure)
            #record the transition from current to next
            transitions[curr_name][symbol] = dfa_state[next_closure]
    
    return dfa_state[start_closure], transitions, accepting,alphabet

In [ ]:
#DFA minimization
def minimize_dfa(start_dfa, transitions, accepting, alphabet):
    # minimization of dfa using moore's algorithm
    dfa_states = set(transitions.keys())
    
    for edges in transitions.values():
        for target in edges.values():
            dfa_states.add(target)
    dfa_states.update(accepting)
    dfa_states.add(start_dfa)

    # initialize the partitions
    dfa_accepting_states = frozenset(accepting)
    dfa_reject_states = frozenset(dfa_states - accepting)

    dfa_state_to_min_dfa_state = {}
    min_dfa_states = []
    
    if dfa_accepting_states:
        min_dfa_states.append(dfa_accepting_states)
        for s in dfa_accepting_states:
            dfa_state_to_min_dfa_state[s] = dfa_accepting_states

    if dfa_reject_states:
        min_dfa_states.append(dfa_reject_states)
        for s in dfa_reject_states:
            dfa_state_to_min_dfa_state[s] = dfa_reject_states

    # refine the partitions until no more refinement is possible
    more_refinement = True
    while more_refinement:
        more_refinement = False
        # We iterate over a copy of the list because we might modify it during the loop
        for min_dfa_state in list(min_dfa_states):
            if len(min_dfa_state) > 1:

                split_states = {}

                for dfa_state in min_dfa_state:
                    # create a signature for the state based on its transitions
                    signature_dict = {}
                    for symbol in alphabet:
                        target = transitions.get(dfa_state, {}).get(symbol)
                        if target:
                            signature_dict[symbol] = dfa_state_to_min_dfa_state[target]
                    signature = tuple(sorted((k,v) for k,v in signature_dict.items()))

                    if signature not in split_states:
                        split_states[signature] = set()
                    split_states[signature].add(dfa_state)

                if len(split_states) > 1:
                    min_dfa_states.remove(min_dfa_state)

                    for splitted_set in split_states.values():
                        splitted_frozenset = frozenset(splitted_set)
                        min_dfa_states.append(splitted_frozenset)

                        for s in splitted_set:
                            dfa_state_to_min_dfa_state[s] = splitted_frozenset
                            
                    more_refinement = True 
                    break
                    
    # Build the Final output JSON Structure
    partition_map = {}
    for i, part in enumerate(min_dfa_states):
        part_name = f"S{i}"
        partition_map[part] = part_name
    
    min_dfa = {}

    start_partition = dfa_state_to_min_dfa_state[start_dfa]
    min_dfa['startingState'] = partition_map[start_partition]
    
    for part in min_dfa_states:
        rep = list(part)[0]  # representative state for this partition
        part_name = partition_map[part]

        min_dfa[part_name] = {
            "isTerminatingState": rep in accepting
        }
        for symbol in alphabet:
            target = transitions.get(rep, {}).get(symbol) 
            if target:
                target_partition = dfa_state_to_min_dfa_state[target]
                min_dfa[part_name][symbol] = partition_map[target_partition]
                
    dead_states = set()
    for state, data in min_dfa.items():
        if state == 'startingState': continue
        if not data["isTerminatingState"]:
            if all(data.get(sym) == state for sym in alphabet): 
                dead_states.add(state)
    
    for ds in dead_states:
        if min_dfa["startingState"] != ds:
            del min_dfa[ds]
            for state, data in min_dfa.items():
                if state == 'startingState': continue
                keys_to_delete = [k for k, v in data.items() if v == ds and k in alphabet]
                for k in keys_to_delete:
                    del data[k]
                    
    return min_dfa, alphabet

In [ ]:
#finally visualization
def visualize_dfa(min_dfa,output_filename = "minimized_dfa"):
    dot = Digraph(name='Minimized DFA')

    #graphs lef to right
    dot.attr(rankdir='LR')
    dot.attr('node', fontname='Helvetica')
    dot.attr('edge', fontname='Helvetica')

    dot.node('start','',shape='none',width='0',height='0')
    dot.edge('start',min_dfa['startingState'])

    for state, data in min_dfa.items():
        if state == 'startingState':
            continue
        shape = 'doublecircle' if data.get('isTerminatingState', False) else 'circle'
        dot.node(state, shape=shape)
    
    for state, data in min_dfa.items():
        if state == 'startingState':
            continue
        targets ={}
        for symbol, val in data.items():
            if symbol != 'isTerminatingState':
                if val not in targets:
                    targets[val] = []
                targets[val].append(symbol)
        for target, symbols in targets.items():
            label = ", ".join(sorted(symbols))
            dot.edge(state, target, label=label)
    dot.render(output_filename, format='png', cleanup=True)
    print(f"graph saved as {output_filename}")
    display(dot)

In [ ]:
#i lied one more cell to tie everything together

nfa_data=load_nfa("nfa_output.json")

#convert
start_dfa, dfa_trans, dfa_accepting, alphabet = nfa_to_dfa(nfa_data)

#minimize
minimized_dfa, alphabet = minimize_dfa(start_dfa, dfa_trans, dfa_accepting, alphabet)

#save to a json file
output_json_path = "minimized_dfa_output.json"
with open(output_json_path, 'w') as f:
    json.dump(minimized_dfa, f, indent=4)
print(f"Minimized DFA successfully saved to {output_json_path}")

#visulize this
visualize_dfa(minimized_dfa)